# Exploración de ventas

## Importación de librerías y datos

In [ ]:
# libraries
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# transformation functions
import dataops_taller_jonatan_palomares.transform as tf

In [ ]:
# data
sys.path.append(str(Path.cwd().parent))
from dataops_taller_jonatan_palomares.extract import extract_data

db_path = Path.cwd().parent / "data" / "database.db"
df = extract_data(db_path)

## Revisión general del conjunto de datos

Vistazo a los primeros registro del conjunto de datos

In [ ]:
df.head()

Hay 5 registros faltantes de la variable `cantidad` y 5 de la variable `precio_unitario`

In [ ]:
df.info()

## Limpieza y transformación de los datos

In [ ]:
# get clean dataframe
clean_df = tf.clean_data(df)

# create additional columns
final_df = tf.calculate_metrics(clean_df)

Vistazo a los datos limpios y con las columnas `venta_total` y `mes` añadidas

In [ ]:
final_df.head()

Tras la limpieza ya no hay valores faltantes

In [ ]:
final_df.info()

## Exploración univariada

In [ ]:
fecha = final_df["fecha"]
print(f"Rango de fechas: {fecha.min().date()} - {fecha.max().date()}")

In [ ]:
print(f"Cantidad de productos únicos: {final_df["producto"].nunique()}")

In [ ]:
# counts per category
category_counts = final_df["categoria"].value_counts().sort_index()
category_counts.plot(kind="bar", figsize=(6, 4), color="#060", edgecolor="#000")
plt.title("Ventas por categoría")
plt.xlabel("Categoría")
plt.ylabel("Número de registros")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# distributions of 'cantidad'
plt.hist(final_df["cantidad"], bins=10, color="#060", edgecolor="#000")
plt.title("Histograma de cantidad")
plt.xlabel("Cantidad")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

In [ ]:
# distributions of 'precio_unitario'
plt.hist(final_df["precio_unitario"], bins=20, color="#060", edgecolor="#000")
plt.title("Histograma de precio_unitario")
plt.xlabel("Precio_unitario")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

## Ventas agregadas

In [ ]:
aggregated_sales_df = tf.aggregate_sales(final_df)

In [ ]:
# total sales by month and category
month_names = [
    "Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio",
    "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"
]
colors = ["#000F08", "#0197F6", "#FBB13C", "#D64045", "#AAAAAA"]

min_month = aggregated_sales_df["mes"].min()
max_month = aggregated_sales_df["mes"].max()

sales_by_month = aggregated_sales_df.pivot(
    index="mes", columns="categoria", values="venta_total"
).reindex(range(min_month, max_month+1), fill_value=0)

sales_by_month.index = [month_names[month - 1] for month in sales_by_month.index]
sales_by_month.plot(kind="line", marker="o", figsize=(8, 4), color=colors)
plt.title("Venta total mensual por categoría")
plt.xlabel("Mes")
plt.ylabel("Venta total")
plt.xticks(rotation=0)
plt.legend(title="Categoría")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()